# Word2Vec

1. Skip-gram

In [ ]:
#6.3 기본 Skip-gram 클래스
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings = vocab_size,
        embedding_dim = embedding_dim
    )
    self.linear = nn.Linear(
        in_features = embedding_dim,
        out_features= vocab_size
    )
  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    output = self.linear(embeddings)
    return output

In [ ]:
!pip install Korpora konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 48.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [konlpy]


In [ ]:
#6.4 영화 리뷰 데이터세트 전처리
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 125MB/s]                             
[nsmc] download ratings_test.txt: 4.90MB [00:00, 49.9MB/s]


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [ ]:
#6.5 단어 사전 구축
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [ ]:
#6.6 Skip-gram의 단어 쌍 추출
def get_word_pairs(tokens, window_size):
  pairs = []
  for sentence in tokens:
    sentence_length = len(sentence)
    for idx, center_word in enumerate(sentence):
      window_start = max(0, idx - window_size)
      window_end = min(sentence_length, idx+window_size + 1)
      center_word = sentence[idx]
      context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
      for context_words in context_words:
        pairs.append([center_word, context_words])
  return pairs

word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [ ]:
#6.7 인덱스 쌍 변환
def get_index_pairs(word_pairs, token_to_id):
  pairs=[]
  unk_index = token_to_id["<unk>"]
  for word_pair in word_pairs:
    center_word, context_word = word_pair
    center_index = token_to_id.get(center_word, unk_index)
    context_index = token_to_id.get(context_word, unk_index)
    pairs.append([center_index, context_index])
  return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


In [ ]:
#6.8 데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexs = index_pairs[:, 0]
context_indexs = index_pairs[:, 1]

dataset = TensorDataset(center_indexs, context_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
#6.9 Skip-gram 모델 준비 작업
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [ ]:
#6.10 모델 학습
for epoch in range(10):
  cost = 0.0
  for input_ids, target_ids in dataloader:
    input_ids = input_ids.to(device)
    target_ids = target_ids.to(device)

    logits = word2vec(input_ids)
    loss = criterion(logits, target_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    cost += loss

  cost = cost / len(dataloader)
  print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.197
Epoch :    2, Cost : 5.982
Epoch :    3, Cost : 5.932
Epoch :    4, Cost : 5.902
Epoch :    5, Cost : 5.880
Epoch :    6, Cost : 5.862
Epoch :    7, Cost : 5.847
Epoch :    8, Cost : 5.835
Epoch :    9, Cost : 5.823
Epoch :   10, Cost : 5.813


In [ ]:
#6.11 임베딩 값 추출
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
  token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-0.47883427  0.85431314 -1.2921654   0.67181206 -0.37834916 -0.5557282
 -0.9012619  -1.3096914  -0.10786772 -0.07467126  1.5240954  -0.4830918
 -1.2399973   0.2718518  -1.0353262  -0.12860358 -1.7037714  -1.479959
 -1.2606106  -0.626833    1.1915538   1.9328616  -1.554156    0.03760043
  1.229016    0.9402982  -0.28830034 -0.23692583  0.5276061  -0.13554972
  0.08986316  0.21467511  0.4492423  -0.43702868 -0.1378558  -0.0887328
  1.3088046   0.21166351 -0.38579503 -1.0586418   0.39055952 -1.6996591
  0.47154647 -1.0006459   0.67153275 -0.4179976   1.3744028   1.3783712
 -0.61703557 -0.18310471 -0.27623186 -1.2044724   0.35602462 -0.35604283
 -0.8711236   0.29241654 -0.28898287  0.58500034  0.8859344  -0.8437389
 -0.53593946  1.1558899   0.60371387 -0.3624527  -0.18122856 -0.55203766
 -0.53100234  0.5979187   0.37132555  0.630611    0.28061727 -0.04293815
 -0.7920264  -0.98929185 -1.0708202   0.35131463 -0.10138053  0.5071445
 -0.29411763  2.1060283  -1.017338    1.8068495   0.55044

In [ ]:
#6.12 단어 임베딩 유사도 계산
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
  cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
  return cosine

def top_n_index(cosine_matrix, n):
  closest_indexes = cosine_matrix.argsort()[::-1]
  top_n = closest_indexes[1 : n+1]
  return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5개 단어")
for index in top_n:
  print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5개 단어
미드 - 유사도 : 0.3185
배우 - 유사도 : 0.3092
견자단 - 유사도 : 0.3047
도중 - 유사도 : 0.2787
도데 - 유사도 : 0.2639


2. Gensim

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.6 MB/s  0:00:00


In [ ]:
#6.13 Word2Vec 모델 학습
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 3,
    max_final_vocab = 10000
)

In [ ]:
#6.14 임베딩 추출 및 유사도 계산
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2='연기력'))

[-0.02043969 -0.22141491  0.496714    0.4835902   0.05815591 -0.25334492
 -0.3323491  -0.0490008  -0.317581    0.39438966  0.1272996  -0.10779907
 -0.38291523  0.10663348 -0.24049607 -0.25866237 -0.26558822  0.23563163
 -0.09330381  0.3036325   0.5556136   0.29849914 -0.20238824  0.02724048
 -0.40569064  0.14737454 -0.42922938  0.26404172  0.23419559 -0.11245949
 -0.25199795  0.03620887  0.05825292 -0.16555981 -0.06065606 -0.07337035
  0.18171872 -0.02990444 -0.07391989 -0.09150232 -0.10171533  0.3170271
 -0.16255213 -0.44871885 -0.16712648  0.12317536 -0.18532681 -0.37537012
  0.13353251  0.11315342  0.5881179   0.27299297  0.26460728 -0.00124219
 -0.4695114  -0.21506026 -0.13487725  0.11181599 -0.21358639 -0.00745539
  0.11860234 -0.23431492  0.0912845   0.14232683 -0.17716944  0.17946516
  0.22586365  0.09290487  0.39625484 -0.26449597 -0.2934193  -0.38674238
 -0.5878083   0.00833224  0.08396015 -0.21665283 -0.3743726  -0.70928574
 -0.23883449  0.25754747 -0.21932517 -0.15384531  0.

# fastText

In [ ]:
#6.15 KorNLI 데이터세트 전처리
from Korpora import Korpora

corpus = Korpora.load('kornli')
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

[kornli] download multinli.train.ko.tsv: 83.6MB [00:00, 112MB/s]                            
[kornli] download snli_1.0_train.ko.tsv: 78.5MB [00:00, 160MB/s]                            
[kornli] download xnli.dev.ko.tsv: 516kB [00:00, 10.3MB/s]
[kornli] download xnli.test.ko.tsv: 1.04MB [00:00, 17.0MB/s]


[['개념적으로', '크림', '스키밍은', '제품과', '지리라는', '두', '가지', '기본', '차원을', '가지고', '있다.'], ['시즌', '중에', '알고', '있는', '거', '알아?', '네', '레벨에서', '다음', '레벨로', '잃어버리는', '거야', '브레이브스가', '모팀을', '떠올리기로', '결정하면', '브레이브스가', '트리플', 'A에서', '한', '남자를', '떠올리기로', '결정하면', '더블', 'A가', '그를', '대신하러', '올라가고', 'A', '한', '명이', '그를', '대신하러', '올라간다.'], ['우리', '번호', '중', '하나가', '당신의', '지시를', '세밀하게', '수행할', '것이다.']]


In [ ]:
#6.16 fastText 모델 실습
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size = 128,
    window=5,
    min_count=5,
    sg = 1,
    epochs = 3,
    min_n = 2,
    max_n = 6
)

In [ ]:
#6.17 fastText OOV 처리
oov_token = '사랑해요'
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn=5))

False
[('사랑해', 0.9010390639305115), ('사랑', 0.8750235438346863), ('사랑한', 0.8688454627990723), ('사랑해.', 0.8427281975746155), ('사랑해서', 0.8358405232429504)]


# 순환 신경망

In [ ]:
#6.20 문장 분류 모델
from torch import nn

class SentenceClassifier(nn.Module):
  def __init__(
      self,
      n_vocab,
      hidden_dim,
      embedding_dim,
      n_layers,
      dropout=0.5,
      bidirectional=True,
      model_type='lstm'
  ):
      super().__init__()

      self.embedding = nn.Embedding(
          num_embeddings=n_vocab,
          embedding_dim=embedding_dim,
          padding_idx=0
      )
      if model_type == 'rnn':
        self.model = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout,
            batch_first=True,
        )
      elif model_type == 'lstm':
        self.model = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout,
            batch_first=True,
        )

      if bidirectional:
        self.classifier = nn.Linear(hidden_dim * 2, 1)
      else:
        self.classifier = nn.Linear(hidden_dim, 1)

      self.dropout = nn.Dropout(dropout)

  def forward(self, inputs):
    embeddings = self.embedding(inputs)
    output, _ = self.model(embeddings)
    last_output = output[:, -1, :]
    last_output = self.dropout(last_output)
    logits = self.classifier(last_output)
    return logits

In [ ]:
#6.21 데이터세트 불러오기
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print('Training Data Size :', len(train))
print('Testing Data Size :', len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
#6.22 데이터 토큰화 및 단어 사전 구축
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=['<pad>', '<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


In [ ]:
#6.23 정수 인코딩 및 패딩
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
  result = list()
  for sequence in sequences:
    sequence = sequence[:max_length]
    pad_length = max_length - len(sequence)
    padded_sequence = sequence + [pad_value] * pad_length
    result.append(padded_sequence)
  return np.asarray(result)

unk_id = token_to_id['<unk>']
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id['<pad>']
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [ ]:
#6.24 데이터로더 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
#6.25 손실 함수와 최적화 함수 정의
from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

In [ ]:
#6.26 모델 학습 및 테스트
def train(model, datasets, criterion, optimizer, device, interval):
  model.train()
  losses = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % interval == 0:
      print(f'Train Loss {step} : {np.mean(losses)}')

def test(model, datasets, criterion, device):
  model.eval()
  losses = list()
  corrects = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())
    yhat = torch.sigmoid(logits)>.5
    corrects.extend(
        torch.eq(yhat, labels).cpu().tolist()
    )

  print(f'Val Loss : {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}')

epochs = 5
interval = 500
for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0 : 0.70316481590271
Train Loss 500 : 0.6908069023353135
Train Loss 1000 : 0.6839204709251205
Train Loss 1500 : 0.6770513206819627
Train Loss 2000 : 0.6677653659349201
Train Loss 2500 : 0.6582999429503521
Val Loss : 0.560157717892918, Val Accuracy : 0.7224
Train Loss 0 : 0.7357432842254639
Train Loss 500 : 0.5280097358360024
Train Loss 1000 : 0.5080021998503587
Train Loss 1500 : 0.49835308100643516
Train Loss 2000 : 0.4901438549362023
Train Loss 2500 : 0.48205899632201105
Val Loss : 0.43577823528466514, Val Accuracy : 0.8038
Train Loss 0 : 0.4427538514137268
Train Loss 500 : 0.3973689873864551
Train Loss 1000 : 0.39854558985252364
Train Loss 1500 : 0.39800868589905086
Train Loss 2000 : 0.39663062207419414
Train Loss 2500 : 0.39380719832649996
Val Loss : 0.3987389266395721, Val Accuracy : 0.819
Train Loss 0 : 0.32107916474342346
Train Loss 500 : 0.33978968568726214
Train Loss 1000 : 0.3398251513635124
Train Loss 1500 : 0.34177754515115616
Train Loss 2000 : 0.344833568416197
T

In [ ]:
#6.27 학습된 모델로부터 임베딩 추출
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
  token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 0.7708984  -0.19784646 -0.7817474  -1.8287921  -0.6960397   1.4851872
  0.13988532  0.27301124 -0.900719    0.11067126  0.57672155  0.7759846
  1.5378001   1.7519845  -1.604663   -0.21262886  1.3406103  -1.376799
 -1.1334493   0.2542596  -0.2955591  -1.259117    0.01562789  0.14721106
 -2.462276   -0.6177872   0.2324047   0.40398142  0.43782824 -0.35291317
 -1.2433717   0.30141324 -1.6718462   0.09117939 -0.20231357  1.2428923
 -0.2838629  -0.599292   -0.60309327 -0.4221396   2.3447375  -0.8251755
 -0.08518945 -0.64443564 -0.4522807   0.18836668  1.5835756   0.59912825
  1.5871254   2.1304526  -0.8034186  -1.4644154  -0.08027658 -0.36101797
  0.5369409  -0.6563603  -0.7238943  -0.3965891  -0.6971958   1.5459237
  1.0249407  -0.58472604 -0.5832003  -1.4989295   0.12077353  0.30517
  2.2615802   0.9321904   0.6732537   0.7546591   1.8531789   0.26571104
 -1.4027272  -0.28516975 -1.3523941   0.38367036  0.33831954  1.4057382
 -0.02684611  0.7940157  -0.22648302  0.9763155   1.24126

In [ ]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    workers=2,
    epochs=5
)

word2vec.save('/content/word2vec.model')

In [ ]:
# 6.28 사전 학습된 모델로 임베딩 계층 초기화
import numpy as np
import torch
from torch import nn
from gensim.models import Word2Vec

word2vec = Word2Vec.load('/content/word2vec.model')
init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
    if token not in ['<pad>', '<unk>'] and token in word2vec.wv:
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)

In [ ]:
#6.29 사전 학습된 임베딩 계층 적용

class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type='lstm',
        pretrained_embedding=None,
    ):
        super().__init__()

        # 사전 학습된 임베딩 적용
        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32)
            )
        else:
            self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx=0
            )

        if model_type == 'rnn':
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == 'lstm':
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits


In [ ]:
#6.30 사전 학습된 임베딩을 사용한 모델 학습
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim,
    n_layers=n_layers, pretrained_embedding=init_embeddings
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500

for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0 : 0.6854491233825684
Train Loss 500 : 0.6742682652083224
Train Loss 1000 : 0.663510112376599
Train Loss 1500 : 0.6426318968478082
Train Loss 2000 : 0.6302549344846333
Train Loss 2500 : 0.6176103247863586
Val Loss : 0.5351591765309294, Val Accuracy : 0.741
Train Loss 0 : 0.5480313301086426
Train Loss 500 : 0.5376854471401302
Train Loss 1000 : 0.5266246418138365
Train Loss 1500 : 0.5213743114892361
Train Loss 2000 : 0.5152642684674513
Train Loss 2500 : 0.5113894328123472
Val Loss : 0.4555537214103979, Val Accuracy : 0.7842
Train Loss 0 : 0.3665771186351776
Train Loss 500 : 0.4789765231802078
Train Loss 1000 : 0.4712613807780878
Train Loss 1500 : 0.46816668808857337
Train Loss 2000 : 0.46684321513895627
Train Loss 2500 : 0.4637968303804777
Val Loss : 0.44059053215736776, Val Accuracy : 0.7888
Train Loss 0 : 0.49772417545318604
Train Loss 500 : 0.4465798759710289
Train Loss 1000 : 0.4476585268051355
Train Loss 1500 : 0.4462128277820877
Train Loss 2000 : 0.44238208967080894
Tra